# Graph Structures Lab

```{contents}
:local:
:depth: 2
```

## Build and Inspect Route Data

```{index} weighted graph; route data; graph contract
```

Use the facilities team's six-room graph to verify the representation before adding search algorithms. Remote is an intentional isolate. Office–Storage has zero rounded minutes, an intentional boundary case. All other route costs are nonnegative, and each route is two-way.

Download {download}`the complete RouteGraph source and verification program <../../materials/19/RouteGraph.cs>` for a console project. Each example below includes its own setup and class so it can run independently.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
foreach (string room in graph.Vertices().OrderBy(r => r, StringComparer.Ordinal))
{
    var neighbors = graph.Neighbors(room).OrderBy(e => e.Key, StringComparer.Ordinal);
    Console.WriteLine($"{room}: {string.Join(", ", neighbors.Select(e => $"{e.Key} ({e.Value})"))}");
}
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, valid={graph.IsValid()}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

## Updates and Boundary Checks

```{index} graph updates; isolated vertex; root-independent structure
```

Graphs have no distinguished root under this model. Any vertex may be removed without deleting the whole structure. Choose update expectations before running the following lifecycle.

| Update | Vertex count | Edge count | Reason |
| --- | ---: | ---: | --- |
| Initial graph | 6 | 4 | Includes Remote and a zero-weight edge. |
| Replace Hall–Lab with five minutes | 6 | 4 | Same endpoints, updated weight. |
| Remove Hall–Office | 6 | 3 | Delete one logical edge in two directions. |
| Remove Hall | 5 | 1 | Remaining Hall–Lobby and Hall–Lab edges disappear. |
| Add Remote–Lobby | 5 | 2 | Previously isolated vertices become adjacent. |

The resulting graph remains disconnected: one component joins Remote/Lobby, one joins Office/Storage, and Lab is isolated. Update correctness is about preserving the stored relationships; a valid graph need not be connected.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
graph.SetEdge("Hall", "Lab", 5);
graph.RemoveEdge("Hall", "Office");
graph.RemoveVertex("Hall");
graph.SetEdge("Remote", "Lobby", 1);
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, valid={graph.IsValid()}");
Console.WriteLine($"Lab degree={graph.Degree("Lab")}, Hall removed again={graph.RemoveVertex("Hall")}");
int edgesBefore = graph.EdgeCount;
try { graph.SetEdge("Lobby", "Missing", 2); }
catch (KeyNotFoundException) { Console.WriteLine("unknown endpoint rejected"); }
try { graph.SetEdge("Lobby", "Lobby", 0); }
catch (ArgumentException) { Console.WriteLine("self-loop rejected"); }
Console.WriteLine($"failed updates preserved edges={graph.EdgeCount == edgesBefore}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

## Export and Compare a Weighted Matrix

```{index} representation conversion; matrix equivalence
```

Export the same graph to a nullable matrix. Choose a label order and keep it with the matrix, because a cell's indexes have no meaning without that mapping. The converter's labels must be exactly the graph's distinct vertices; the example obtains them from a vertex snapshot.

Visit every stored adjacency entry and fill its cell. All other cells stay null. An isolate produces an empty row and column. The converter uses O(V²) time/space for array initialization plus O(V + E) work to enumerate the graph. The final representation still has matrix cost, even though the source is sparse.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
string[] labels = graph.Vertices().OrderBy(r => r, StringComparer.Ordinal).ToArray();
int?[,] matrix = ToMatrix(graph, labels);
for (int i = 0; i < labels.Length; i++)
    for (int j = 0; j < labels.Length; j++)
    {
        bool found = graph.TryGetWeight(labels[i], labels[j], out int weight);
        if (matrix[i, j].HasValue != found || (found && matrix[i, j] != weight))
            throw new Exception("Graph/matrix mismatch.");
        if (matrix[i, j] != matrix[j, i]) throw new Exception("Asymmetric matrix.");
    }
int office = Array.IndexOf(labels, "Office"), storage = Array.IndexOf(labels, "Storage");
Console.WriteLine($"matrix cells={matrix.Length}, equivalence passed");
Console.WriteLine($"Office-Storage exists={matrix[office, storage].HasValue}, minutes={matrix[office, storage]}");

int?[,] ToMatrix(RouteGraph graph, string[] labels)
{
    var index = labels.Select((label, i) => (label, i)).ToDictionary(x => x.label, x => x.i);
    var matrix = new int?[labels.Length, labels.Length];
    foreach (string vertex in labels)
        foreach (var edge in graph.Neighbors(vertex))
            matrix[index[vertex], index[edge.Key]] = edge.Value;
    return matrix;
}

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

Pair-by-pair comparison checks both presence and weight. Comparing only non-null weights could miss an unintended edge; treating zero as absence could miss the Office–Storage route. Matrix symmetry is necessary for this undirected model, but symmetry alone does not establish that all intended edges survived conversion.

## Validate a Supplied Walk

```{index} walk validation; weighted cost; direct edge
```

A walk checker receives a route chosen elsewhere. It does not discover a route or prove optimality. Validate each label and each consecutive edge, then sum the weights. A repeated vertex is legal for a walk, so a return trip contributes its edges again.

Use a long total to accumulate integer weights without overflowing an int on a long route. Return false for an empty route, an unknown vertex, or a missing direct edge. A known singleton route succeeds with cost zero. On failure, leave the output at zero rather than returning a partial trip cost.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
foreach (string[] route in new[]
{
    new[] { "Lobby", "Hall", "Office", "Storage" },
    new[] { "Lobby", "Hall", "Lobby" },
    new[] { "Remote" }, new[] { "Lobby", "Storage" },
    new[] { "Missing" }, Array.Empty<string>()
})
{
    bool valid = TryWalkCost(graph, route, out long minutes);
    Console.WriteLine($"[{string.Join(", ", route)}]: valid={valid}, minutes={minutes}");
}

bool TryWalkCost(RouteGraph graph, string[] route, out long minutes)
{
    minutes = 0;
    if (route.Length == 0) return false;
    var vertices = new HashSet<string>(graph.Vertices(), StringComparer.Ordinal);
    if (route.Any(v => !vertices.Contains(v))) return false;
    long total = 0;
    for (int i = 1; i < route.Length; i++)
    {
        if (!graph.TryGetWeight(route[i - 1], route[i], out int weight)) return false;
        total += weight;
    }
    minutes = total;
    return true;
}

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

The valid examples cost seven, eight, and zero minutes. The remaining examples are invalid. The first is a path; the second is a walk that revisits Lobby; the third demonstrates an isolated vertex. None of these results says that a route between two endpoints is shortest.

This wrapper snapshots V labels to check singleton existence, so setup takes O(V) time/space. Checking a supplied route of L labels then takes expected O(L) time under the hash assumptions. A graph API with direct vertex membership could avoid that snapshot; this implementation makes the tradeoff visible rather than hiding it.

## Exercise

```{index} graph verification; exercise
```

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
// Exercise: remove Lab, preserve the other vertices, and check degree sum.
// Your code starts here.
/* TODO: remove Lab and compute the remaining degree sum. */
// Your code ends here.
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, valid={graph.IsValid()}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var graph = new RouteGraph();
foreach (string room in new[] { "Lobby", "Hall", "Lab", "Office", "Storage", "Remote" }) graph.AddVertex(room);
graph.SetEdge("Lobby", "Hall", 4);
graph.SetEdge("Hall", "Lab", 2);
graph.SetEdge("Hall", "Office", 3);
graph.SetEdge("Office", "Storage", 0);
// Solution
graph.RemoveVertex("Lab");
int degreeSum = graph.Vertices().Sum(v => graph.Degree(v));
Console.WriteLine($"degree sum={degreeSum}, twice edges={2 * graph.EdgeCount}");
Console.WriteLine($"vertices={graph.VertexCount}, edges={graph.EdgeCount}, valid={graph.IsValid()}");

public class RouteGraph
{
    private readonly Dictionary<string, Dictionary<string, int>> adjacency
        = new(StringComparer.Ordinal);
    public int VertexCount => adjacency.Count;
    public int EdgeCount { get; private set; }

    public bool AddVertex(string vertex)
        => adjacency.TryAdd(vertex, new Dictionary<string, int>(StringComparer.Ordinal));

    public bool SetEdge(string a, string b, int minutes)
    {
        if (a == b) throw new ArgumentException("Self-loops are not allowed.");
        if (minutes < 0) throw new ArgumentOutOfRangeException(nameof(minutes));
        RequireVertex(a);
        RequireVertex(b);
        bool added = !adjacency[a].ContainsKey(b);
        adjacency[a][b] = minutes;
        adjacency[b][a] = minutes;
        if (added) EdgeCount++;
        return added;
    }

    public bool TryGetWeight(string a, string b, out int minutes)
    {
        minutes = default;
        return adjacency.TryGetValue(a, out var neighbors)
            && neighbors.TryGetValue(b, out minutes);
    }

    public int Degree(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].Count;
    }

    public KeyValuePair<string, int>[] Neighbors(string vertex)
    {
        RequireVertex(vertex);
        return adjacency[vertex].ToArray();
    }

    public string[] Vertices() => adjacency.Keys.ToArray();

    public bool RemoveEdge(string a, string b)
    {
        if (!adjacency.TryGetValue(a, out var neighbors) || !neighbors.Remove(b)) return false;
        adjacency[b].Remove(a);
        EdgeCount--;
        return true;
    }

    public bool RemoveVertex(string vertex)
    {
        if (!adjacency.TryGetValue(vertex, out var neighbors)) return false;
        foreach (string neighbor in neighbors.Keys) adjacency[neighbor].Remove(vertex);
        EdgeCount -= neighbors.Count;
        adjacency.Remove(vertex);
        return true;
    }

    public bool IsValid()
    {
        long degreeSum = 0;
        foreach (var (vertex, neighbors) in adjacency)
        {
            degreeSum += neighbors.Count;
            foreach (var (neighbor, minutes) in neighbors)
            {
                if (vertex == neighbor || minutes < 0
                    || !adjacency.TryGetValue(neighbor, out var reverse)
                    || !reverse.TryGetValue(vertex, out int reverseMinutes)
                    || reverseMinutes != minutes) return false;
            }
        }
        return degreeSum == 2L * EdgeCount;
    }

    private void RequireVertex(string vertex)
    {
        if (!adjacency.ContainsKey(vertex)) throw new KeyNotFoundException($"Unknown vertex: {vertex}");
    }
}

degree sum=6, twice edges=6
vertices=5, edges=3, valid=True


## Summary

```{index} graph representation; verification
```

Graph storage must preserve its model rules across every update. Keep isolates, update both directions for undirected edges, distinguish zero from absence, and compare representations by both membership and weight. Validate a supplied walk before summing it; leave route discovery and optimization to the later algorithm chapters.

```{rubric} Footnotes
```

[^1]: Chapter 23 uses weighted graphs for greedy optimization; Chapter 25 develops traversal and reachability. This chapter supplies their representation foundation.